# Simple Gemini tests

### From Youtube

In [ ]:
import os
from google.adk import types

: 

In [5]:
! pip install python-dotenv

You should consider upgrading via the '/Users/rifatmahammod/.pyenv/versions/3.10.2/bin/python3.10 -m pip install --upgrade pip' command.


In [ ]:
from dotenv import load_dotenv

from google import genai
from google.genai import types

load_dotenv(dotenv_path='../.env')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')    
client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model='models/gemini-2.5-flash-lite',
    contents=types.Content(
        parts=[
            types.Part(
                file_data=types.FileData(file_uri='https://www.youtube.com/shorts/nsdlYMDpXbU')
            ),
            types.Part(text='Please summarize the video in 3 sentences and extracts 3 key timestamps with descriptions from the video')
        ]
    )
)

In [8]:
response.parts[0].text

'The video shows a man making a doner kebab sandwich. He grinds beef with onion, yogurt, garlic, paprika, salt, and pepper, then flattens it into a thin sheet and cooks it. Finally, he wraps the cooked meat with tomatoes, jalapeños, a creamy sauce, and pomegranate molasses in a pita bread.\n\nHere are 3 key timestamps with descriptions from the video:\n\n*   **00:06 - 00:17:** The man begins by adding blended onion and yogurt to ground beef in a bowl, along with seasonings like garlic, paprika, salt, and pepper. He then mixes everything together thoroughly with gloved hands.\n*   **00:22 - 00:27:** He demonstrates how to flatten the meat mixture into a thin, even layer between two sheets of parchment paper using a rolling pin, aiming for a thin sheet that can be easily rolled.\n*   **00:53 - 01:03:** The process of assembling the sandwich is shown, where the cooked meat is placed on a pita bread, followed by tomatoes, jalapeños, a creamy sauce, pomegranate molasses, and sumac.'

### From File

In [3]:
from google import genai

client = genai.Client()

myfile = client.files.upload(file="../recipe_analyzer/outputs/nsdlYMDpXbU.mp4")

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents=[myfile, "Summarize this video in 3 sentences."]
)

print(response.text)

ClientError: 400 FAILED_PRECONDITION. {'error': {'code': 400, 'message': 'The File uwr3gquclz1j is not in an ACTIVE state and usage is not allowed.', 'status': 'FAILED_PRECONDITION'}}

### Constructing Video Analyzer Agent

In [ ]:
"""Video Analyzer Agent - Analyzes YouTube cooking videos using Gemini"""

import logging
from google.adk.agents import Agent
from google import genai
from google.genai import types
import os
from pathlib import Path
import time

logger = logging.getLogger(__name__)


def analyze_video_with_gemini(video_url: str) -> dict:
    """
    Analyze a cooking video using Gemini's multimodal capabilities.

    Args:
        video_url: Original YouTube URL

    Returns:
        Dictionary with extracted recipe information
    """
    try:
        # Initialize Gemini client
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("GEMINI_API_KEY environment variable not set")

        client = genai.Client(api_key=api_key)

        # Upload video file
        # logger.info(f"Uploading video: {video_path}")
        # video_file = client.files.upload(file=video_path)

        # Create prompt for recipe extraction
        prompt = """
        Analyze this cooking video and extract all recipe information. Provide:

        1. Recipe title
        2. List of ALL ingredients mentioned (with approximate quantities if visible/mentioned)
        3. Step-by-step cooking instructions in order
        4. Estimated number of servings
        5. Estimated preparation time (in minutes)
        6. Estimated cooking time (in minutes)
        7. Difficulty level (Easy, Medium, or Hard)
        8. Cuisine type (if identifiable)
        9. Any cooking tips or tricks mentioned
        10. Tags (e.g., vegetarian, vegan, gluten-free, quick, healthy, halal)

        Pay close attention to:
        - Visual cues showing ingredients
        - Any text overlays with measurements
        - Spoken instructions
        - Timing of different steps

        Format your response as a structured analysis.
        """

        # Generate content with video
        logger.info("Analyzing video with Gemini...")
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=types.Content(
                parts=[
                    types.Part(
                        file_data=types.FileData(file_uri=video_url),
                        video_metadata=types.VideoMetadata(fps=0.1)
                    ),
                    types.Part(text=prompt)
                        ]
                )
            )

        analysis_text = response.text

        # Parse the response and extract structured data
        # For now, return the raw analysis
        return {
            "video_url": video_url,
            "analysis": analysis_text,
            # "video_file_id": video_file.name if hasattr(video_file, 'name') else None
        }

    except Exception as e:
        logger.error(f"Video analysis failed: {e}")
        raise RuntimeError(f"Failed to analyze video: {str(e)}")


# Create video analyzer agent
video_analyzer_agent = Agent(
    name="video_analyzer",
    model="gemini-2.5-flash-lite",
    description="Analyzes YouTube cooking videos to extract recipe information using multimodal AI.",
    instruction="""
    You are a video analyzer specialized in cooking videos.

    Your task is to:
    1. Take a YouTube Shorts URL as input. 
    2. Analyze the video content using Gemini's multimodal capabilities using analyze_video_with_gemini tool
    3. Extract all recipe information including ingredients, steps, timing, and metadata

    When analyzing:
    - Pay attention to both visual and audio cues
    - Extract exact quantities when visible or mentioned
    - Note the sequence of cooking steps
    - Identify cooking techniques and tips
    - Detect dietary restrictions or cuisine types

    Always use the analyze_video_with_gemini tool to process videos.
    Return a comprehensive analysis with all extracted information.
    """,
    tools=[analyze_video_with_gemini],
    output_key="recipe_analysis"
)

# initialize session and runner to test the agent

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

session_service = InMemorySessionService()
await session_service.create_session(
    app_name="app", user_id="test_user", session_id="test_session"
)
runner = Runner(
    agent=video_analyzer_agent, app_name="app", session_service=session_service
    )
query = "https://www.youtube.com/shorts/nsdlYMDpXbU"
async for event in runner.run_async(
    user_id="test_user",
    session_id="test_session",
    new_message=genai_types.Content(
        role="user", 
        parts=[genai_types.Part.from_text(text=query)]
    ),
):
    if event.is_final_response() and event.content and event.content.parts:
        print(event.content.parts[0].text)

Here is a detailed analysis of the "Doner Kebab Sandwich" recipe from the video:

### Doner Kebab Sandwich

**1. Recipe Title:** Doner Kebab Sandwich

**2. List of ALL Ingredients:**
*   **Ground Beef:** Approximately 1-1.5 lbs (visual estimate)
*   **Onion:** 1 medium (grated or blended)
*   **Yogurt:** 1/2 tablespoon
*   **Garlic:** Generous seasoning (to taste)
*   **Paprika Chili:** Generous seasoning (to taste)
*   **Salt:** To taste
*   **Pepper:** To taste
*   **Tomatoes:** 1-2 (sliced, for baking with meat and for sandwich assembly)
*   **Jalapeños:** 1-2 (sliced, for baking with meat and for sandwich assembly)
*   **Pita Bread:** 1 piece per sandwich (approx. 4-5 pieces)
*   **Arthena Sauce:** "Lots" (to taste, for internal and external application)
*   **Pomegranate Molasses:** "A little" (to taste)
*   **Sumac:** "Lots" (to taste, optional)

**3. Step-by-Step Cooking Instructions:**
1.  **Prepare the Meat Mixture:** Grate or blend one onion and add it to the ground beef. Add